# Spike 01 — PDF Image Extraction

**Goal:** Download both versions of the Madinah Tranquil City report and extract every page as a clean PNG image.

**Time box:** 1 hour

**Question to answer:** Can we extract clean, OCR-ready page images from these PDFs?

**Done when:** `data/images_en/` and `data/images_ar/` contain sharp PNG images of every page.

---

### Why PyMuPDF (fitz)?
- No external dependencies (no Poppler install needed on Windows)
- Renders pages directly using the PDF's vector data — no quality loss
- Fast: processes a page in milliseconds
- Returns pixel data we control (DPI, colorspace, format)

## Step 1 — Imports

In [ ]:
import fitz          # PyMuPDF — pip install PyMuPDF
import requests
import os
from pathlib import Path

print(f"PyMuPDF version : {fitz.version[0]}")
print("✅ Imports OK")

## Step 2 — Download PDFs

Both URLs come from `muo.mda.gov.sa/reports-en.html` and the Arabic equivalent.  
Files go into `../data/pdfs/` (relative to this notebook's location).

In [ ]:
PDF_DIR = Path("../data/pdfs")
PDF_DIR.mkdir(parents=True, exist_ok=True)

PDFS = {
    "tranquil_en.pdf": "https://muo.mda.gov.sa/download/Madinah - the Tranquil Livable City 2024.pdf",
    "tranquil_ar.pdf": "https://muo.mda.gov.sa/download/\u0627\u0644\u0645\u062f\u064a\u0646\u0629 \u0627\u0644\u0645\u0646\u0648\u0631\u0629 - \u0645\u062f\u064a\u0646\u0629 \u0627\u0644\u0633\u0643\u064a\u0646\u0629 \u0627\u0644\u0642\u0627\u0628\u0644\u0629 \u0644\u0644\u0639\u064a\u0634 2024_Mobile_3.pdf"
}

for filename, url in PDFS.items():
    save_path = PDF_DIR / filename

    if save_path.exists():
        print(f"⏭  Already downloaded: {filename} ({save_path.stat().st_size // 1024} KB)")
        continue

    print(f"⬇  Downloading {filename}...")
    try:
        # Some servers need a browser-like User-Agent
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, headers=headers, timeout=60)
        r.raise_for_status()
        save_path.write_bytes(r.content)
        print(f"   ✅ Saved — {save_path.stat().st_size // 1024} KB")
    except Exception as e:
        print(f"   ❌ Failed: {e}")
        print(f"   Try downloading manually and saving to: {save_path.resolve()}")

## Step 3 — Inspect PDFs Before Extracting

Before converting, let's see what we're dealing with:  
- How many pages?
- Is the text embedded (selectable) or is it a scanned image?

> **Why it matters:** If text is already embedded in the PDF, we might skip OCR entirely for some pages.  
> If pages are scanned images, OCR is mandatory.

In [ ]:
for filename in PDFS.keys():
    pdf_path = PDF_DIR / filename
    if not pdf_path.exists():
        print(f"❌ {filename} not found — run Step 2 first")
        continue

    doc = fitz.open(str(pdf_path))

    print(f"\n{'='*50}")
    print(f"File    : {filename}")
    print(f"Pages   : {len(doc)}")
    print(f"Encrypted: {doc.is_encrypted}")

    # Check if the first 3 pages have embedded text
    for i in range(min(3, len(doc))):
        page = doc[i]
        text = page.get_text().strip()
        word_count = len(text.split())
        has_text = word_count > 10
        print(f"  Page {i+1}: {'✅ has embedded text' if has_text else '🖼  image only (OCR needed)'} ({word_count} words)")
        if has_text:
            # Show a snippet of the embedded text
            print(f"    Preview: {text[:100]}...")

    doc.close()

## Step 4 — Extract Pages as PNG Images

### DPI Choice — 150 DPI
| DPI | File size | OCR quality | Use case |
|-----|-----------|-------------|----------|
| 72  | Tiny      | Poor        | Thumbnails only |
| 150 | Medium    | **Good** ✅ | **Our choice — OCR APIs** |
| 300 | Large     | Excellent   | Print-quality archiving |

150 DPI keeps API payload sizes manageable while giving OCR models enough resolution to read small text and table cells.

In [ ]:
def extract_pages_as_images(pdf_path: str, output_dir: str, dpi: int = 150) -> list:
    """
    Converts every page of a PDF into a PNG image.

    Returns a list of saved image file paths.

    Args:
        pdf_path  : path to the PDF file
        output_dir: folder to save PNG images into
        dpi       : resolution (150 is the sweet spot for OCR APIs)
    """
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(pdf_path)
    saved_paths = []

    print(f"Extracting {len(doc)} pages from {Path(pdf_path).name} at {dpi} DPI...")

    for i, page in enumerate(doc):
        # get_pixmap renders the page as a raster image
        # matrix scales the page — fitz.Matrix(dpi/72, dpi/72) converts to target DPI
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)

        img_path = out / f"page_{i+1:03d}.png"   # zero-padded: page_001, page_002...
        pix.save(str(img_path))
        saved_paths.append(img_path)

        size_kb = img_path.stat().st_size // 1024
        print(f"  Page {i+1:>3} → {img_path.name}  ({pix.width}x{pix.height}px, {size_kb} KB)")

    doc.close()
    print(f"Done. {len(saved_paths)} images saved to {out.resolve()}\n")
    return saved_paths


# Extract English pages
en_images = extract_pages_as_images(
    pdf_path   = str(PDF_DIR / "tranquil_en.pdf"),
    output_dir = "../data/images_en",
    dpi        = 150
)

# Extract Arabic pages
ar_images = extract_pages_as_images(
    pdf_path   = str(PDF_DIR / "tranquil_ar.pdf"),
    output_dir = "../data/images_ar",
    dpi        = 150
)

## Step 5 — Visual Check (display sample pages)

Always look at the output before moving on. Bad images = bad OCR = bad RAG.

In [ ]:
from IPython.display import display, Image as IPImage

print("=== ENGLISH — Page 1 ===")
if en_images:
    display(IPImage(filename=str(en_images[0]), width=600))

print("\n=== ARABIC — Page 1 ===")
if ar_images:
    display(IPImage(filename=str(ar_images[0]), width=600))

In [ ]:
# Show a page that likely has a table (usually page 3-5 in these reports)
print("=== ENGLISH — Page 4 (likely has a table) ===")
if len(en_images) >= 4:
    display(IPImage(filename=str(en_images[3]), width=700))

## Step 6 — Summary

Run this cell to see if the spike passed.

In [ ]:
en_dir = Path("../data/images_en")
ar_dir = Path("../data/images_ar")

en_count = len(list(en_dir.glob("*.png")))
ar_count = len(list(ar_dir.glob("*.png")))

print("=" * 50)
print("SPIKE 01 — RESULTS")
print("=" * 50)
print(f"English images extracted : {en_count}")
print(f"Arabic images extracted  : {ar_count}")
print()

if en_count > 0 and ar_count > 0:
    print("✅ SPIKE PASSED — Images ready for Spike 02 and 03")
    print(f"   Next: Run spike_02_arabic_ocr.ipynb on images in {ar_dir.resolve()}")
    print(f"   Next: Run spike_03_english_ocr.ipynb on images in {en_dir.resolve()}")
else:
    print("❌ SPIKE FAILED")
    if en_count == 0:
        print("   English PDF may not have downloaded — check data/pdfs/")
    if ar_count == 0:
        print("   Arabic PDF may not have downloaded — check data/pdfs/")